# Exercise 5.5 — An open cylinder (tube) among the cubes

**Algorithm.**
1. Choose $n$ (here 24). For $k = 0 \dots n$, angle $a_k = 2\pi k / n$.
2. Bottom circle points: $(\cos a_k, -1, \sin a_k)$; top circle points: $(\cos a_k, 1, \sin a_k)$.
   (The point $k = n$ is the same as $k = 0$, so the ring closes.)
3. For each $k$, the four points bottom$_k$, bottom$_{k+1}$, top$_k$, top$_{k+1}$ form a small rectangle, which we
   split into **2 triangles**: (b$_k$, t$_k$, t$_{k+1}$) and (b$_k$, t$_{k+1}$, b$_{k+1}$).
4. No triangles for the top and bottom, so the tube is **open**.

Total: $n \times 2 \times 3 = 144$ vertices for $n = 24$. The more points, the rounder the tube.

The tube has its own buffers (created once) and is drawn with the same shader and camera as the cubes.
We also tilt it and turn it slowly so you can see inside the open ends.

## Library
Only **glm-js** is needed (for matrices). The WebGL code is written directly, so this notebook still works
even when the course JS server is down.

In [ ]:
%%html
<script src='https://git.io/glm-js.min.js'></script>

In [ ]:
%%html
<script id="ex55-vs" type="x-shader/x-vertex">
precision mediump float;
attribute vec3 in_Position;
attribute vec3 in_Color;
uniform mat4 MVP;
varying vec3 ex_Color;
void main(void) {
    gl_Position = MVP * vec4(in_Position, 1.0);
    ex_Color = in_Color;
}
</script>

In [ ]:
%%html
<script id="ex55-fs" type="x-shader/x-fragment">
precision mediump float;
varying vec3 ex_Color;
void main(void) {
    gl_FragColor = vec4(ex_Color, 1.0);
}
</script>

In [ ]:
%%html
<script>
// ---------- small WebGL helpers ----------
// compile the two shaders (read from the <script> tags) and link them into a program
window.makeProgram = function (gl, vsId, fsId) {
  const compile = (type, id) => {
    const s = gl.createShader(type);
    gl.shaderSource(s, document.getElementById(id).text);
    gl.compileShader(s);
    if (!gl.getShaderParameter(s, gl.COMPILE_STATUS)) console.error(id, gl.getShaderInfoLog(s));
    return s;
  };
  const p = gl.createProgram();
  gl.attachShader(p, compile(gl.VERTEX_SHADER, vsId));
  gl.attachShader(p, compile(gl.FRAGMENT_SHADER, fsId));
  gl.linkProgram(p);
  return p;
};
// copy a JS array of numbers into a GPU buffer (GL wants 32-bit floats)
window.makeBuffer = function (gl, data) {
  const b = gl.createBuffer();
  gl.bindBuffer(gl.ARRAY_BUFFER, b);
  gl.bufferData(gl.ARRAY_BUFFER, new Float32Array(data), gl.STATIC_DRAW);
  return b;
};
// connect a buffer (3 floats per vertex) to an attribute of the shader
window.useAttrib = function (gl, prog, name, buffer) {
  const loc = gl.getAttribLocation(prog, name);
  if (loc < 0) return;
  gl.bindBuffer(gl.ARRAY_BUFFER, buffer);
  gl.vertexAttribPointer(loc, 3, gl.FLOAT, false, 0, 0);
  gl.enableVertexAttribArray(loc);
};
// call render(time) on every animation frame
window.animate = function (render) {
  const step = (time) => { render(time); requestAnimationFrame(step); };
  requestAnimationFrame(step);
};
// ---------- cube: 6 faces x 2 triangles x 3 vertices = 36 vertices ----------
// only coordinates -1 and 1 are used; the 8 corners are repeated for every face
window.cubeData = function () {
  const p = [[-1,-1,-1], [ 1,-1,-1], [ 1, 1,-1], [-1, 1,-1],
             [-1,-1, 1], [ 1,-1, 1], [ 1, 1, 1], [-1, 1, 1]];
  // one different colour per corner (red, green, blue, yellow, ...)
  const c = [[1,0,0], [0,1,0], [0,0,1], [1,1,0], [1,0,1], [0,1,1], [1,0.5,0], [0.5,0.5,0.5]];
  const faces = [[4,5,6,7], [1,0,3,2], [0,4,7,3], [5,1,2,6], [7,6,2,3], [0,1,5,4]];
  const pos = [], col = [];
  for (const [a, b, cc, d] of faces)
    for (const i of [a, b, cc, a, cc, d]) {      // square face = 2 triangles
      pos.push(...p[i]);
      col.push(...c[i]);
    }
  return { pos, col, n: pos.length / 3 };
};
// ---------- open cylinder (tube) ----------
// 1) vertices: n points on a circle at y=-1 and n points at y=+1
// 2) triangles: points k and k+1 of the bottom and top circles form a square = 2 triangles
// No caps, so it is open at both ends.
window.tubeData = function (n) {
  const bottom = [], top = [];
  for (let k = 0; k <= n; k++) {                     // k = n repeats k = 0 to close the ring
    const a = 2 * Math.PI * k / n;
    bottom.push([Math.cos(a), -1, Math.sin(a)]);
    top.push([Math.cos(a), 1, Math.sin(a)]);
  }
  const pos = [], col = [];
  const colour = (p) => [0.5 + 0.5 * p[0], 0.3, 0.5 + 0.5 * p[2]];   // colour changes around the tube
  for (let k = 0; k < n; k++) {
    const b0 = bottom[k], b1 = bottom[k + 1], t0 = top[k], t1 = top[k + 1];
    for (const p of [b0, t0, t1,   b0, t1, b1]) { pos.push(...p); col.push(...colour(p)); }
  }
  return { pos, col, n: pos.length / 3 };
};
</script>

In [ ]:
%%html
<canvas id="ex55Canvas" width="500" height="500" style="border:2px solid #000000;"></canvas>
<div id="ex55CanvasInfo" style="font-family:monospace"></div>
<script>
(function () {
  const cv = document.getElementById("ex55Canvas");
  const gl = cv.getContext("webgl");
  const prog = makeProgram(gl, "ex55-vs", "ex55-fs");
  const cube = cubeData(), tube = tubeData(24);
  const cubePos = makeBuffer(gl, cube.pos), cubeCol = makeBuffer(gl, cube.col);
  const tubePos = makeBuffer(gl, tube.pos), tubeCol = makeBuffer(gl, tube.col);
  const uMVP = gl.getUniformLocation(prog, "MVP");
  gl.enable(gl.DEPTH_TEST);
  gl.clearColor(1, 1, 1, 1);

  // each cube: position + rotation axis + speed. All use the SAME two buffers.
  const cubes = [
    { pos: [-3,  0,  0], axis: [1, 0, 0], speed: 1.0 },
        { pos: [ 3,  0,  0], axis: [0, 0, 1], speed: 1.3 },
    { pos: [-1.5, 2.5, -2], axis: [1, 1, 0], speed: 0.9 },
    { pos: [ 1.5, -2.5, -2], axis: [0, 1, 1], speed: 1.6 },
  ];

  const V = glm.lookAt(glm.vec3(0, 3, 12), glm.vec3(0, 0, 0), glm.vec3(0, 1, 0));
  const P = glm.perspective(glm.radians(45), cv.width / cv.height, 0.1, 100);

  animate((time) => {
    gl.clear(gl.COLOR_BUFFER_BIT | gl.DEPTH_BUFFER_BIT);
    gl.useProgram(prog);

    // cubes
    useAttrib(gl, prog, "in_Position", cubePos);
    useAttrib(gl, prog, "in_Color", cubeCol);
    for (const c of cubes) {
    const M = glm.translate(glm.mat4(1), glm.vec3(...c.pos))['*'](
              glm.rotate(glm.mat4(1), c.speed * time / 1000, glm.normalize(glm.vec3(...c.axis))));
      gl.uniformMatrix4fv(uMVP, false, P['*'](V['*'](M)).elements);
      gl.drawArrays(gl.TRIANGLES, 0, cube.n);
    }

    // tube in the middle: tilted 60 degrees and turning
    useAttrib(gl, prog, "in_Position", tubePos);
    useAttrib(gl, prog, "in_Color", tubeCol);
    const M = glm.rotate(glm.mat4(1), time / 1500, glm.vec3(0, 1, 0))['*'](
              glm.rotate(glm.mat4(1), glm.radians(60), glm.vec3(1, 0, 0)))['*'](
              glm.scale(glm.mat4(1), glm.vec3(1, 1.5, 1)));
    gl.uniformMatrix4fv(uMVP, false, P['*'](V['*'](M)).elements);
    gl.drawArrays(gl.TRIANGLES, 0, tube.n);

    document.getElementById("ex55CanvasInfo").textContent = "open tube: " + tube.n + " vertices, plus 4 cubes";
  });
})();
</script>